# Inspect VDS

In [1]:
%%configure -f
{
    "driverMemory": "45G"
}

In [2]:
# Import and initiate HAIL
import hail as hl
hl.init(sc,log='/tmp/hail.log')

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2,application_1755759866455_0003,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.5.2-amzn-1
SparkUI available at http://ip-192-168-108-36.ap-southeast-1.compute.internal:38869
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /tmp/hail.log

In [3]:
from pprint import pprint
pprint(dict(hl.spark_context().getConf().getAll()))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{'spark.app.attempt.id': '1',
 'spark.app.id': 'application_1755759866455_0003',
 'spark.app.name': 'livy-session-2',
 'spark.app.startTime': '1755768300108',
 'spark.app.submitTime': '1755768288008',
 'spark.blacklist.decommissioning.enabled': 'true',
 'spark.blacklist.decommissioning.timeout': '1h',
 'spark.decommissioning.timeout.threshold': '20',
 'spark.default.parallelism': '1008',
 'spark.driver.defaultJavaOptions': "-XX:OnOutOfMemoryError='kill -9 %p'",
 'spark.driver.extraClassPath': '/usr/local/lib/python3.9/site-packages/hail/backend/hail-all-spark.jar:/usr/lib/hadoop-lzo/lib/*:/usr/lib/hadoop/hadoop-aws.jar:/usr/share/aws/aws-java-sdk/*:/usr/share/aws/emr/emrfs/conf:/usr/share/aws/emr/emrfs/lib/*:/usr/share/aws/emr/emrfs/auxlib/*:/usr/share/aws/emr/goodies/lib/emr-spark-goodies.jar:/usr/share/aws/emr/security/conf:/usr/share/aws/emr/security/lib/*:/usr/share/aws/hmclient/lib/aws-glue-datacatalog-spark-client.jar:/usr/share/java/Hive-JSON-Serde/hive-openx-serde.jar:/usr/shar

In [4]:
# source

vds_prefix = 's3://precise-scratch/goypav/1KG/VDS/'

# input
vds_uri = vds_prefix + '1000genomes_combined_batch1_2_3_4.bf2-tr500k-sp1k.n3205.vds'

# output
vd_variant_light_uri = vds_prefix + 'vd_variant_light.mt'
vds_split_multi_uri = vds_prefix + '1000genomes_combined.n3205.splitmulti.vds'
sparse_sample_allrows_mt_uri = "1000genomes_combined.n3205.sparse.sample_allrows.mt"
sparse_sample_variants_only_mt_uri = "1000genomes_combined.n3205.sparse.sample_variants_only.mt"

vd_filt_uri = vds_prefix + "vd_entries_filtered.mt"
vd_sanit_uri = vds_prefix + "vd_likelihoods_sanitized.mt"

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
vds = hl.vds.read_vds(vds_uri)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
vds.reference_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    'ref_block_max_length': int32
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
----------------------------------------
Entry fields:
    'LEN': int32
    'DP': int32
    'GQ': int32
    'ICNT': array<int32>
    'MIN_DP': int32
    'SPL': array<int32>
    'LGT': call
    'LAD': array<int32>
    'END': int32
----------------------------------------
Column key: ['s']
Row key: ['locus']
----------------------------------------

In [7]:
hl.eval(vds.reference_data.ref_block_max_length)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

151393

In [8]:
vds.variant_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'LA': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LPL': array<int32>
    'RGQ': int32
    'gvcf_info': struct {
        DB: bool, 
        FS: float64, 
        FractionInformativeReads: float64, 
        LOD: float64, 
        MQ: float64, 
        MQRankSum: float64, 
        QD: float64, 
        R2_5P_bias: float64, 
        ReadPosRankSum: float64, 
        SOR: float64
    }
    'AF': array<float64>
    'DP': int32
    'F1R2': array<int32>
    'F2R1': array<int32>
    'GP': array<float64>
    'GQ': int32
    'ICNT': array<int32>
    'MB': array<int32>
    'MIN_DP': int32
    'PRI': array<float64>
    'PS': int32
    'SB': array<int32>
    'SPL': array<int32

In [9]:
# Count rows/cols in the variant_data MT
print(f"Reference genome: {vds.variant_data.locus.dtype.reference_genome.name}")
print(f"Number of samples: {vds.n_samples()}")
print(f"Number of variant partitions: {vds.variant_data.n_partitions()}")
print(f"Total number of variants: {vds.variant_data.count_rows():,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Reference genome: GRCh38
Number of samples: 3205
Number of variant partitions: 5767
Total number of variants: 156,228,032

# QC tests

In [10]:

def vds_nano_qc(vds, p=0.003, coalesce=64, tmp_path=None, seed=1):
    """
    Lightweight QC for small clusters (no densify).
    - p: row sampling fraction (e.g. 0.003 = 0.3%)
    - coalesce: reduce partitions without shuffle
    - tmp_path: optional checkpoint path to speed repeats
    """
    hl.set_global_seed(seed)

    # Keep only row keys (locus, alleles) and drop entry fields
    vd = vds.variant_data.select_rows().select_entries()

    # Fewer tasks for an 8-CPU cluster
    if coalesce:
        vd = vd.naive_coalesce(coalesce)

    # Optional: cache slim view for repeated QC runs
    if tmp_path:
        vd = vd.checkpoint(tmp_path, _read_if_exists=True)

    # Smoke test – should start tasks immediately
    vd.rows().show(5)

    # Tiny random sample for quick aggregates
    sampled = vd.filter_rows(hl.rand_bool(p))

    agg = sampled.aggregate_rows(hl.struct(
        total = hl.agg.count(),
        bi    = hl.agg.count_where(hl.len(sampled.alleles) == 2),

        # Type mix (based on first alt)
        snp = hl.agg.count_where(hl.is_snp(sampled.alleles[0], sampled.alleles[1])),
        ins = hl.agg.count_where(hl.len(sampled.alleles[1]) > hl.len(sampled.alleles[0])),
        dele = hl.agg.count_where(hl.len(sampled.alleles[1]) < hl.len(sampled.alleles[0])),
        mnv = hl.agg.count_where(
            (hl.len(sampled.alleles[1]) == hl.len(sampled.alleles[0])) &
            (hl.len(sampled.alleles[0]) > 1)
        ),

        # Ti/Tv
        ti = hl.agg.count_where(
            hl.is_snp(sampled.alleles[0], sampled.alleles[1]) &
            hl.is_transition(sampled.alleles[0], sampled.alleles[1])
        ),
        tv = hl.agg.count_where(
            hl.is_snp(sampled.alleles[0], sampled.alleles[1]) &
            ~hl.is_transition(sampled.alleles[0], sampled.alleles[1])
        ),

        # Minimal representation
        not_minrep = hl.agg.count_where(
            (hl.min_rep(sampled.locus, sampled.alleles).locus != sampled.locus) |
            (hl.min_rep(sampled.locus, sampled.alleles).alleles != sampled.alleles)
        ),

        # Contig distribution
        contigs = hl.agg.counter(sampled.locus.contig),
    ))

    scale = int(round(1 / p))
    est_total = agg.total * scale
    est_bi    = agg.bi    * scale
    est_multi = est_total - est_bi

    print("\n≈Row-level QC (scaled from sample):")
    print(f"Estimated total variants: {est_total:,}")
    print(f"Estimated biallelic: {est_bi:,}")
    print(f"Estimated multiallelic: {est_multi:,} ({est_multi/est_total*100:.2f}%)")

    print("\nVariant type mix (approx):")
    for label, val in {
        "SNP": agg.snp, "INS": agg.ins, "DEL": agg.dele, "MNV": agg.mnv
    }.items():
        print(f"  {label}: {val*scale:,}")

    if agg.tv > 0:
        print(f"\nTi/Tv (approx): {agg.ti/agg.tv:.3f}  "
              f"(Ti ~{agg.ti*scale:,}, Tv ~{agg.tv*scale:,})")
    else:
        print("\nTi/Tv: not enough SNPs in sample")

    print(f"\nNon-minimal-representation sites (approx): {agg.not_minrep*scale:,} "
          f"({agg.not_minrep/agg.total*100:.2f}% of sampled)")

    print("\nTop contigs by count (approx):")
    for contig, cnt in sorted(agg.contigs.items(), key=lambda x: -x[1])[:10]:
        print(f"  {contig}: {cnt*scale:,}")

# Example:
# vds = hl.vds.read_vds(vds_path)
# vds_nano_qc(vds, p=0.003, coalesce=64, tmp_path="s3://YOUR-BUCKET/tmp/vd_variant_light.mt")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
vds_nano_qc(vds, p=0.003, coalesce=64, tmp_path=vd_variant_light_uri)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------------+-------------------+
| locus         | alleles           |
+---------------+-------------------+
| locus<GRCh38> | array<str>        |
+---------------+-------------------+
| chr1:10001    | ["T","A","C","G"] |
| chr1:10013    | ["TA","T"]        |
| chr1:10091    | ["T","A"]         |
| chr1:10097    | ["T","A"]         |
| chr1:10103    | ["T","A"]         |
+---------------+-------------------+
showing top 5 rows


?Row-level QC (scaled from sample):
Estimated total variants: 156,185,991
Estimated biallelic: 136,155,708
Estimated multiallelic: 20,030,283 (12.82%)

Variant type mix (approx):
  SNP: 141,252,939
  INS: 4,727,601
  DEL: 10,205,451
  MNV: 2,161,836

Ti/Tv (approx): 1.348  (Ti ~81,081,837, Tv ~60,171,102)

Non-minimal-representation sites (approx): 0 (0.00% of sampled)

Top contigs by count (approx):
  chr1: 12,556,764
  chr2: 12,171,150
  chr3: 9,846,477
  chr4: 9,630,027
  chr7: 9,029,295
  chr5: 8,744,913
  chr11: 8,484,507
  chr6: 8,230,428
  chr8: 7,

In [14]:
# check metrics
vd = vds.variant_data

# --- 1) Biallelic vs multiallelic split (no genotypes needed) ---
allele_counts = vd.aggregate_rows(hl.agg.counter(hl.len(vd.alleles)))
n_biallelic = allele_counts.get(2, 0)
n_multiallelic = sum(c for k, c in allele_counts.items() if k and k > 2)
print(f"Biallelic variants: {n_biallelic:,}")
print(f"Multiallelic variants: {n_multiallelic:,} "
      f"({(n_multiallelic/(n_biallelic+n_multiallelic))*100:.2f}% of sites)")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Biallelic variants: 136,256,201
Multiallelic variants: 19,971,831 (12.78% of sites)

In [16]:
# --- 2) Variant type breakdown (SNP / INS / DEL / MNV / OTHER) ---

# (if not already slimmed)
# vd = vds.variant_data.select_rows().select_entries()

has_alt = hl.len(vd.alleles) > 1
ref = vd.alleles[0]
alt = hl.or_missing(has_alt, vd.alleles[1])  # safe if any rows lack an ALT

# Use nested if_else (avoids the IR bug with hl.case())
var_type_expr = hl.if_else(
    ~has_alt, "OTHER",
    hl.if_else(
        hl.is_snp(ref, alt), "SNP",
        hl.if_else(
            hl.len(alt) > hl.len(ref), "INS",
            hl.if_else(
                hl.len(alt) < hl.len(ref), "DEL",
                hl.if_else((hl.len(alt) == hl.len(ref)) & (hl.len(ref) > 1), "MNV", "OTHER")
            )
        )
    )
)

type_counts = vd.aggregate_rows(hl.agg.counter(var_type_expr))
for k in ["SNP","INS","DEL","MNV","OTHER"]:
    print(f"{k}: {type_counts.get(k, 0):,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SNP: 141,257,601
INS: 4,761,480
DEL: 10,208,951
MNV: 0
OTHER: 0

In [18]:
# --- 3) Ti/Tv ratio (on SNPs only; from alleles, not genotypes) ---
# (fixed: avoid source mismatch by doing everything in one aggregate)

has_alt = hl.len(vd.alleles) > 1
ref = vd.alleles[0]
alt = vd.alleles[1]

is_snp = has_alt & hl.is_snp(ref, alt)
is_ti  = is_snp & hl.is_transition(ref, alt)

counts_ti_tv = vd.aggregate_rows(hl.struct(
    ti = hl.agg.count_where(is_ti),
    tv = hl.agg.count_where(is_snp & ~hl.is_transition(ref, alt)),
))

if counts_ti_tv.tv > 0:
    print(f"Transitions: {counts_ti_tv.ti:,}, Transversions: {counts_ti_tv.tv:,}, "
          f"Ti/Tv: {counts_ti_tv.ti / counts_ti_tv.tv:.3f}")
else:
    print("No transversions found (tv=0).")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Transitions: 81,214,007, Transversions: 60,043,594, Ti/Tv: 1.353

In [19]:
# --- 4) Contig distribution (top 25 contigs by count) ---
contig_counts = vd.aggregate_rows(hl.agg.counter(vd.locus.contig))
# sort by count desc and print a few
for contig, cnt in sorted(contig_counts.items(), key=lambda x: -x[1])[:25]:
    print(f"{contig}: {cnt:,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

chr1: 12,402,488
chr2: 12,230,294
chr3: 9,769,440
chr4: 9,693,704
chr7: 9,004,242
chr5: 8,776,702
chr11: 8,450,703
chr6: 8,153,921
chr8: 7,827,815
chr12: 7,500,615
chr9: 7,295,152
chr10: 7,190,927
chrX: 6,367,130
chr13: 5,638,286
chr18: 5,556,044
chr16: 5,102,817
chr15: 4,566,211
chr17: 4,294,496
chr14: 4,248,772
chr20: 3,963,927
chr19: 3,114,725
chr22: 2,239,044
chr21: 2,173,845
chrY: 654,903
chrM: 11,829

In [20]:
# --- 5) Left-normalization / minimal representation check ---
minrep = hl.min_rep(vd.locus, vd.alleles)
n_not_minrep = vd.aggregate_rows(
    hl.agg.count_where((minrep.locus != vd.locus) | (minrep.alleles != vd.alleles))
)
print(f"Non-minimal-representation sites: {n_not_minrep:,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Non-minimal-representation sites: 0

In [21]:
# --- 6) Optional: rsID presence rate if field exists ---
row_fields = set(vd.row_value.dtype.fields)
if "rsid" in row_fields:
    n_with_rsid = vd.aggregate_rows(hl.agg.count_where(hl.is_defined(vd.rsid) & (vd.rsid != "")))
    n_total     = vds.variant_data.count_rows()
    print(f"Sites with rsID: {n_with_rsid:,} ({n_with_rsid/n_total:.2%})")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Sites with rsID: 0 (0.00%)

In [28]:
# --- MQ quick scan (lift one defined entry to row) ---
# Use the full variant_data (NOT the slimmed vd)
vd_full = vds.variant_data

# entries table with MQ
et1 = vd_full.select_entries(MQ = hl.float64(vd_full.gvcf_info.MQ)).entries()
# rebind after filter
et2 = et1.filter(hl.is_defined(et1.MQ))

row_mq = et2.group_by(et2.locus, et2.alleles).aggregate(
    MQ_row = hl.agg.take(et2.MQ, 1)[0]
)

mq_stats = row_mq.aggregate(hl.struct(
    n   = hl.agg.count(),
    mean= hl.agg.mean(row_mq.MQ_row),
    qs  = hl.agg.approx_quantiles(row_mq.MQ_row, [0.05, 0.50, 0.95]),
))
p5, p50, p95 = mq_stats.qs
    # commas & 2dp
print(f"gvcf_info.MQ (site-level) n={mq_stats.n:,} | "
      f"mean {mq_stats.mean:.2f} | p5 {p5:.2f} | median {p50:.2f} | p95 {p95:.2f}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

gvcf_info.MQ (site-level) n=156,228,032 | mean 194.22 | p5 5.00 | median 250.00 | p95 250.00
2025-08-13 08:26:36.825 Hail: INFO: Coerced sorted dataset

In [24]:
vd.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'LA': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LPL': array<int32>
    'RGQ': int32
    'gvcf_info': struct {
        DB: bool, 
        FS: float64, 
        FractionInformativeReads: float64, 
        LOD: float64, 
        MQ: float64, 
        MQRankSum: float64, 
        QD: float64, 
        R2_5P_bias: float64, 
        ReadPosRankSum: float64, 
        SOR: float64
    }
    'AF': array<float64>
    'DP': int32
    'F1R2': array<int32>
    'F2R1': array<int32>
    'GP': array<float64>
    'GQ': int32
    'ICNT': array<int32>
    'MB': array<int32>
    'MIN_DP': int32
    'PRI': array<float64>
    'PS': int32
    'SB': array<int32>
    'SPL': array<int32

# Test split multiallelic

In [31]:
# 1) Split multiallelics (sparse, no densify)
if not hasattr(hl.vds, "split_multi"):
    raise RuntimeError("This Hail build lacks hl.vds.split_multi; use the dense path instead.")
vds_bi = hl.vds.split_multi(vds)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# 2) (Optional) reduce #output files before write
#    Pick a target shard count that matches your cluster/S3 needs.
#    Comment out if you prefer original partitioning.
# target_shards = 2048
# vd_coalesced = vds_bi.variant_data.naive_coalesce(target_shards)
# vds_bi = hl.vds.VariantDataset(vds_bi.reference_data, vd_coalesced)

In [32]:
# 3) Write biallelic VDS
vds_bi.write(vds_split_multi_uri)#, overwrite=True)
print("Wrote biallelic VDS to:", vds_split_multi_uri)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
Error summary: HailException: PL cannot have missing elements.
------------
Hail stack trace:
  File "/mnt/yarn/usercache/livy/appcache/application_1755069438175_0001/container_1755069438175_0001_01_000001/tmp/4326303865289014712", line 744, in <module>
    sys.exit(main())

  File "/mnt/yarn/usercache/livy/appcache/application_1755069438175_0001/container_1755069438175_0001_01_000001/tmp/4326303865289014712", line 716, in main
    response = handler(content)

  File "/mnt/yarn/usercache/livy/appcache/application_1755069438175_0001/container_1755069438175_0001_01_000001/tmp/4326303865289014712", line 326, in execute_request
    result = node.execute()

  File "/mnt/yarn/usercache/livy/appcache/application_1755069438175_0001/container_1755069438175_0001_01_000001/tmp/4326303865289014712", line 237, in execute
    exec(code, global_dict)

  File "<stdin>", line 5, in <module>

  File "/usr/local/lib/python3.9/site-packages/hail/vds/methods.py", line 645, in spli

In [ ]:
# 4) Quick post-write sanity check (read back fast)
vds_check = hl.vds.read_vds(vds_split_multi_uri)
vd = vds_check.variant_data
n_multi = vd.aggregate_rows(hl.agg.count_where(hl.len(vd.alleles) > 2))
n_rows  = vd.count_rows()
n_samp  = vds_check.n_samples()
print(f"Samples: {n_samp:,}")
print(f"Rows after split: {n_rows:,}")
print(f"Multiallelic rows after split: {n_multi:,}")
assert n_multi == 0, "Found multiallelic rows after split"

In [11]:
vd = vds.variant_data  # sparse MT

def count_faulty_array(mt, field: str):
    if field not in mt.entry.dtype.fields:
        print(f"{field}: not present in entry schema")
        return
    arr = mt[field]
    stats = mt.aggregate_entries(hl.struct(
        n_entries = hl.agg.count(),  # total (non-missing) entries in this sparse MT
        n_defined = hl.agg.count_where(hl.is_defined(arr)),
        n_faulty  = hl.agg.count_where(hl.is_defined(arr) & hl.any(lambda x: hl.is_missing(x), arr)),
    ))
    print(f"{field}: defined {stats.n_defined:,} / total {stats.n_entries:,} entries")
    pct = (stats.n_faulty / stats.n_defined * 100) if stats.n_defined else 0.0
    print(f"{field}: entries with missing elements = {stats.n_faulty:,} ({pct:.4f}% of defined)")

count_faulty_array(vd, "LPL")
count_faulty_array(vd, "PL")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

LPL: defined 17,691,041,658 / total 17,691,273,109 entries
LPL: entries with missing elements = 0 (0.0000% of defined)
PL: not present in entry schema

In [17]:

vd = vds.variant_data  # sparse MT

# ---- helpers (REF-excluded expectations) ----
la      = hl.or_else(vd.LA, hl.empty_array(hl.tint32))
k       = hl.len(la)  # number of local ALTs (no REF)
# ploidy for math (default 2 when LGT missing)
ploidy  = hl.or_else(hl.or_missing(hl.is_defined(vd.LGT), vd.LGT.ploidy), 2)
# ploidy key for grouping (use -1 sentinel so keys are always ints)
ploidy_key = hl.or_else(
    hl.if_else(hl.is_defined(vd.LGT), vd.LGT.ploidy, hl.missing(hl.tint32)),
    hl.int32(-1)
)

# expected LPL length for non-REF genotype likelihoods
exp_len_local = hl.if_else(
    ploidy == 1, k,
    hl.if_else(ploidy == 2, (k * (k + 1)) // 2, hl.missing(hl.tint32))
)

# conditions
faulty         = hl.is_defined(vd.LPL) & hl.any(lambda x: hl.is_missing(x), vd.LPL)
mismatch_local = hl.is_defined(vd.LPL) & hl.is_defined(exp_len_local) & (hl.len(vd.LPL) != exp_len_local)
has_symb       = hl.any(lambda a: (a == "*") | (hl.str(a).startswith("<")), vd.alleles[1:])

# ---- 1) Overall ----
overall = vd.aggregate_entries(hl.struct(
    defined  = hl.agg.count_where(hl.is_defined(vd.LPL)),
    faulty   = hl.agg.count_where(faulty),
    mismatch = hl.agg.count_where(mismatch_local),
    both     = hl.agg.count_where(mismatch_local & faulty),
))
print(f"LPL defined: {overall.defined:,}")
print(f"LPL with missing elements: {overall.faulty:,} "
      f"({(overall.faulty / overall.defined * 100 if overall.defined else 0):.4f}%)")
print(f"LPL length mismatch (REF-excluded expectation): {overall.mismatch:,} | "
      f"both mismatch+faulty: {overall.both:,}")

# ---- 2) By ploidy (faulty & mismatch), -1 = missing ploidy ----
by_ploidy_faulty = vd.aggregate_entries(hl.agg.group_by(ploidy_key, hl.agg.count_where(faulty)))
by_ploidy_mm     = vd.aggregate_entries(hl.agg.group_by(ploidy_key, hl.agg.count_where(mismatch_local)))
print("By ploidy:")
for p in sorted(by_ploidy_faulty.keys()):
    label = "missing" if p == -1 else str(p)
    f = by_ploidy_faulty[p]
    m = by_ploidy_mm.get(p, 0)
    print(f"  ploidy={label}: faulty {f:,} | mismatch {m:,}")

# ---- 3) By number of local ALTs (len(LA) = k) ----
by_k_faulty = vd.aggregate_entries(hl.agg.group_by(k, hl.agg.count_where(faulty)))
by_k_mm     = vd.aggregate_entries(hl.agg.group_by(k, hl.agg.count_where(mismatch_local)))
print("By len(LA):")
for kk in sorted(by_k_faulty.keys())[:10]:
    f = by_k_faulty[kk]
    m = by_k_mm.get(kk, 0)
    print(f"  len(LA)={kk}: faulty {f:,} | mismatch {m:,}")

# ---- 4) Symbolic / '*' rows ----
sym_faulty = vd.aggregate_entries(hl.agg.count_where(faulty & has_symb))
sym_mm     = vd.aggregate_entries(hl.agg.count_where(mismatch_local & has_symb))
print(f"On rows with '*' or symbolic ALT -> faulty: {sym_faulty:,} | mismatch: {sym_mm:,}")

# ---- 5) Peek a few entries (keep keys; show non-key payload) ----
vd.filter_entries(hl.is_defined(vd.LPL) & (ploidy == 2) & (k > 0)) \
  .entries() \
  .select('LA','LGT','LPL','GQ') \
  .show(5)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

LPL defined: 17,691,041,658
LPL with missing elements: 0 (0.0000%)
LPL length mismatch (REF-excluded expectation): 5,523 | both mismatch+faulty: 0
By ploidy:
  ploidy=missing: faulty 0 | mismatch 5,523
  ploidy=1: faulty 0 | mismatch 0
  ploidy=2: faulty 0 | mismatch 0
By len(LA):
  len(LA)=2: faulty 0 | mismatch 0
  len(LA)=3: faulty 0 | mismatch 332
  len(LA)=4: faulty 0 | mismatch 641
  len(LA)=5: faulty 0 | mismatch 867
  len(LA)=6: faulty 0 | mismatch 957
  len(LA)=7: faulty 0 | mismatch 2,726
On rows with '*' or symbolic ALT -> faulty: 0 | mismatch: 0
+---------------+-------------------+-----------+--------------+------+
| locus         | alleles           | s         | LA           | LGT  |
+---------------+-------------------+-----------+--------------+------+
| locus<GRCh38> | array<str>        | str       | array<int32> | call |
+---------------+-------------------+-----------+--------------+------+
| chr1:10001    | ["T","A","C","G"] | "HG00142" | [0,3]        | 0/0  |
| ch

In [16]:

vd = vds.variant_data

la     = hl.or_else(vd.LA, hl.empty_array(hl.tint32))
k      = hl.len(la)                     # number of local ALTs (no REF)
ploidy = hl.or_else(hl.or_missing(hl.is_defined(vd.LGT), vd.LGT.ploidy), 2)

# expected LPL length for non-REF genotypes only
exp_len_local = hl.if_else(
    ploidy == 1, k,
    hl.if_else(ploidy == 2, (k * (k + 1)) // 2, hl.missing(hl.tint32))
)

mismatch_local = hl.is_defined(vd.LPL) & hl.is_defined(exp_len_local) & (hl.len(vd.LPL) != exp_len_local)

mm = vd.aggregate_entries(hl.struct(
    defined = hl.agg.count_where(hl.is_defined(vd.LPL)),
    mismatch_local = hl.agg.count_where(mismatch_local),
))
print(f"LPL defined: {mm.defined:,} | length-mismatched (non-REF expectation): {mm.mismatch_local:,}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

LPL defined: 17,691,041,658 | length-mismatched (non-REF expectation): 5,523

In [18]:
vd = vds.variant_data
la = hl.or_else(vd.LA, hl.empty_array(hl.tint32))
m  = hl.len(la)

ploidy_known = hl.is_defined(vd.LGT)
exp_hap = m
exp_dip = (m * (m + 1)) // 2

is_unknown = ~ploidy_known & hl.is_defined(vd.LPL)
shape_hap = vd.aggregate_entries(hl.agg.count_where(is_unknown & (hl.len(vd.LPL) == exp_hap)))
shape_dip = vd.aggregate_entries(hl.agg.count_where(is_unknown & (hl.len(vd.LPL) == exp_dip)))

print(f"Unknown-ploidy entries shaped as HAPLOID: {shape_hap:,}")
print(f"Unknown-ploidy entries shaped as DIPLOID: {shape_dip:,}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Unknown-ploidy entries shaped as HAPLOID: 5,523
Unknown-ploidy entries shaped as DIPLOID: 865,337

In [19]:
vd = vds.variant_data
la = hl.or_else(vd.LA, hl.empty_array(hl.tint32))
m  = hl.len(la)
ploidy_known = hl.is_defined(vd.LGT)
lenLPL = hl.len(vd.LPL)

exp_hap = m
exp_dip = (m * (m + 1)) // 2
exp_tri = (m * (m + 1) * (m + 2)) // 6  # unordered triploid combos

unknown = ~ploidy_known & hl.is_defined(vd.LPL)
triploid_like = unknown & (lenLPL == exp_tri)
n_tri = vd.aggregate_entries(hl.agg.count_where(triploid_like))
print(f"Unknown-ploidy entries with triploid-shaped LPL: {n_tri:,}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Unknown-ploidy entries with triploid-shaped LPL: 0

In [20]:
vd = vds.variant_data
la = hl.or_else(vd.LA, hl.empty_array(hl.tint32))
m  = hl.len(la)
lenLPL = hl.len(vd.LPL)
ploidy_known = hl.is_defined(vd.LGT)
unknown = ~ploidy_known & hl.is_defined(vd.LPL)

exp_hap = m
exp_dip = (m * (m + 1)) // 2
exp_tri = (m * (m + 1) * (m + 2)) // 6
exp_tet = (m * (m + 1) * (m + 2) * (m + 3)) // 24

counts = vd.aggregate_entries(hl.struct(
    hap = hl.agg.count_where(unknown & (lenLPL == exp_hap)),
    dip = hl.agg.count_where(unknown & (lenLPL == exp_dip)),
    tri = hl.agg.count_where(unknown & (lenLPL == exp_tri)),
    tet = hl.agg.count_where(unknown & (lenLPL == exp_tet)),
    other = hl.agg.count_where(unknown & ~(
        (lenLPL == exp_hap) | (lenLPL == exp_dip) | (lenLPL == exp_tri) | (lenLPL == exp_tet)
    )),
))
print(counts)

# Where do any triploid-like cases occur? (by contig)
by_contig = vd.group_rows_by(contig=vd.locus.contig).aggregate_entries(
    tri = hl.agg.count_where(unknown & (lenLPL == exp_tri)),
    unk = hl.agg.count_where(unknown)
)
by_contig.order_by(hl.desc(by_contig.tri)).show(10)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
GroupedMatrixTable instance has no field, method, or property 'order_by'
    Hint: use 'describe()' to show the names of all data fields.
Traceback (most recent call last):
  File "/usr/local/lib/python3.9/site-packages/hail/table.py", line 171, in __getattr__
    raise AttributeError(get_nice_attr_error(self, item))
AttributeError: GroupedMatrixTable instance has no field, method, or property 'order_by'
    Hint: use 'describe()' to show the names of all data fields.



In [21]:
vd = vds.variant_data

# ---------- Helpers ----------
la       = hl.or_else(vd.LA, hl.empty_array(hl.tint32))
m        = hl.len(la)                               # |LA| including REF
lenLPL   = hl.len(vd.LPL)
has_LPL  = hl.is_defined(vd.LPL)
int_na   = has_LPL & hl.any(lambda x: hl.is_missing(x), vd.LPL)  # internal NA elements (rare)

ploidy_known = hl.is_defined(vd.LGT)
ploidy       = hl.or_else(hl.or_missing(ploidy_known, vd.LGT.ploidy), hl.missing(hl.tint32))
unknown      = ~ploidy_known & has_LPL
known        =  ploidy_known & has_LPL

# Expected REF-inclusive genotype counts by ploidy p
exp_hap = m
exp_dip = (m * (m + 1)) // 2
exp_tri = (m * (m + 1) * (m + 2)) // 6
exp_tet = (m * (m + 1) * (m + 2) * (m + 3)) // 24

# ---------- Buckets ----------
# Known ploidy (Hail LGT provided)
known_hap_good = known & (vd.LGT.ploidy == 1) & (lenLPL == exp_hap)
known_hap_bad  = known & (vd.LGT.ploidy == 1) & (lenLPL != exp_hap)

known_dip_good = known & (vd.LGT.ploidy == 2) & (lenLPL == exp_dip)
known_dip_bad  = known & (vd.LGT.ploidy == 2) & (lenLPL != exp_dip)

known_other_pl = known & ~((vd.LGT.ploidy == 1) | (vd.LGT.ploidy == 2))  # e.g., 0,3,...

# Unknown ploidy (no LGT), infer from shape
unk_hap = unknown & (lenLPL == exp_hap)
unk_dip = unknown & (lenLPL == exp_dip)
unk_tri = unknown & (lenLPL == exp_tri)
unk_tet = unknown & (lenLPL == exp_tet)
unk_oth = unknown & ~( (lenLPL == exp_hap) | (lenLPL == exp_dip) | (lenLPL == exp_tri) | (lenLPL == exp_tet) )

# "Risky for split" heuristic:
#  - any internal NA, or
#  - length that matches none of the accepted shapes given (known/unknown) ploidy
risky_known = int_na | known_hap_bad | known_dip_bad | known_other_pl
risky_unk   = int_na | unk_oth
risky_any   = risky_known | risky_unk

# ---------- One-pass counts ----------
stats = vd.aggregate_entries(hl.struct(
    total            = hl.agg.count(),
    lpl_defined      = hl.agg.count_where(has_LPL),
    lpl_missing      = hl.agg.count_where(~has_LPL),
    internal_na      = hl.agg.count_where(int_na),

    known_hap_good   = hl.agg.count_where(known_hap_good),
    known_hap_bad    = hl.agg.count_where(known_hap_bad),
    known_dip_good   = hl.agg.count_where(known_dip_good),
    known_dip_bad    = hl.agg.count_where(known_dip_bad),
    known_other_pl   = hl.agg.count_where(known_other_pl),

    unk_hap          = hl.agg.count_where(unk_hap),
    unk_dip          = hl.agg.count_where(unk_dip),
    unk_tri          = hl.agg.count_where(unk_tri),
    unk_tet          = hl.agg.count_where(unk_tet),
    unk_oth          = hl.agg.count_where(unk_oth),

    risky_any        = hl.agg.count_where(risky_any),
))

def pct(x, denom): 
    return (x / denom * 100) if denom else 0.0

print("=== LPL shape & risk summary ===")
print(f"Total entries:                {stats.total:,}")
print(f"LPL defined / missing:        {stats.lpl_defined:,} / {stats.lpl_missing:,}")
print(f"Internal NA elements:         {stats.internal_na:,} ({pct(stats.internal_na, stats.lpl_defined):.6f}% of defined)")

print("\nKnown ploidy (from LGT):")
print(f"  hap good / bad:             {stats.known_hap_good:,} / {stats.known_hap_bad:,}")
print(f"  dip good / bad:             {stats.known_dip_good:,} / {stats.known_dip_bad:,}")
print(f"  other ploidy (known):       {stats.known_other_pl:,}")

print("\nUnknown ploidy (no LGT), by shape:")
print(f"  hap-shaped:                 {stats.unk_hap:,}")
print(f"  dip-shaped:                 {stats.unk_dip:,}")
print(f"  triploid-shaped:            {stats.unk_tri:,}")
print(f"  tetraploid-shaped:          {stats.unk_tet:,}")
print(f"  other (none of above):      {stats.unk_oth:,}")

print(f"\nRisky for split (heuristic):  {stats.risky_any:,} "
      f"({pct(stats.risky_any, stats.lpl_defined):.6f}% of LPL-defined)")

# ---------- Example rows per non-empty category ----------
def show_examples(title, cond, n=5):
    # Keep keys; show non-key payload + helpful derived columns
    print(f"\n=== Examples: {title} ===")
    e = vd.filter_entries(cond).entries()
    e = e.select(
        'LA','LGT','LPL','GQ',
        m = hl.len(hl.or_else(e.LA, hl.empty_array(hl.tint32))),
        lenLPL = hl.len(e.LPL)
    )
    e.show(n)

examples = [
    ("INTERNAL NA inside LPL", int_na),
    ("KNOWN ploidy: hap BAD (len != m)", known_hap_bad),
    ("KNOWN ploidy: dip BAD (len != m*(m+1)/2)", known_dip_bad),
    ("KNOWN ploidy: other (not 1 or 2)", known_other_pl),
    ("UNKNOWN ploidy: hap-shaped", unk_hap),
    ("UNKNOWN ploidy: dip-shaped", unk_dip),
    ("UNKNOWN ploidy: triploid-shaped", unk_tri),
    ("UNKNOWN ploidy: tetraploid-shaped", unk_tet),
    ("UNKNOWN ploidy: OTHER (none of the above)", unk_oth),
]

# Only show categories that actually have rows (lightweight check via count_where)
for name, cond in examples:
    cnt = vd.aggregate_entries(hl.agg.count_where(cond))
    if cnt > 0:
        show_examples(f"{name}  (n={cnt:,})", cond, n=min(5, cnt))


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== LPL shape & risk summary ===
Total entries:                17,691,273,109
LPL defined / missing:        17,691,041,658 / 231,451
Internal NA elements:         0 (0.000000% of defined)

Known ploidy (from LGT):
  hap good / bad:             209,436,690 / 0
  dip good / bad:             17,480,734,108 / 0
  other ploidy (known):       0

Unknown ploidy (no LGT), by shape:
  hap-shaped:                 5,523
  dip-shaped:                 865,337
  triploid-shaped:            0
  tetraploid-shaped:          0
  other (none of above):      0

Risky for split (heuristic):  0 (0.000000% of LPL-defined)

=== Examples: UNKNOWN ploidy: hap-shaped  (n=5,523) ===
+---------------+
| locus         |
+---------------+
| locus<GRCh38> |
+---------------+
| chrX:2856799  |
| chrX:2856799  |
| chrX:3012540  |
| chrX:3012603  |
| chrX:3037056  |
+---------------+

+------------------------------------------------------------------------------+
| alleles                                               

In [22]:
# --- classify shapes again (unknown ploidy only) ---
vd = vds.variant_data
la = hl.or_else(vd.LA, hl.empty_array(hl.tint32))
m  = hl.len(la)
lenLPL = hl.len(vd.LPL)

ploidy_known = hl.is_defined(vd.LGT)
unknown = ~ploidy_known & hl.is_defined(vd.LPL)

exp_hap = m
exp_dip = (m * (m + 1)) // 2

unk_hap = unknown & (lenLPL == exp_hap)
unk_dip = unknown & (lenLPL == exp_dip)

# --- check RGQ presence; missing RGQ is the usual cause of a missing 0/0 PL cell ---
n_uh_rgq_miss = vd.aggregate_entries(hl.agg.count_where(unk_hap & hl.is_missing(vd.RGQ)))
n_ud_rgq_miss = vd.aggregate_entries(hl.agg.count_where(unk_dip & hl.is_missing(vd.RGQ)))
print(f"Unknown-ploidy hap-shaped WITH RGQ MISSING: {n_uh_rgq_miss:,}")
print(f"Unknown-ploidy dip-shaped WITH RGQ MISSING: {n_ud_rgq_miss:,}")

# Optional: peek a few of those
vd.filter_entries(unk_hap & hl.is_missing(vd.RGQ)) \
  .entries().select('LA','LGT','LPL','GQ','RGQ').show(5)

# --- minimal, targeted sanitize before split (only where RGQ is missing) ---
# vd_clean = vd.annotate_entries(
#     LPL = hl.cond( (unk_hap | unk_dip) & hl.is_missing(vd.RGQ),
#                    hl.missing(hl.tarray(hl.tint32)),
#                    vd.LPL)
# )
# vds_clean = hl.vds.VariantDataset(vds.reference_data, vd_clean)

# # Now split safely
# vds_bi = hl.vds.split_multi(vds_clean, filter_changed_loci=True)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Unknown-ploidy hap-shaped WITH RGQ MISSING: 0
Unknown-ploidy dip-shaped WITH RGQ MISSING: 0
+---------------+------------+-----+--------------+------+--------------+
| locus         | alleles    | s   | LA           | LGT  | LPL          |
+---------------+------------+-----+--------------+------+--------------+
| locus<GRCh38> | array<str> | str | array<int32> | call | array<int32> |
+---------------+------------+-----+--------------+------+--------------+
+---------------+------------+-----+--------------+------+--------------+

+-------+-------+
|    GQ |   RGQ |
+-------+-------+
| int32 | int32 |
+-------+-------+
+-------+-------+

In [11]:
# Probe: unknown-ploidy entries at multi-ALT rows where the sample only has a subset of ALTs
# --- counts (unchanged) ---
vd = vds.variant_data
la  = hl.or_else(vd.LA, hl.empty_array(hl.tint32))
m   = hl.len(la)
lenLPL = hl.len(vd.LPL)

is_multi_row     = hl.len(vd.alleles) > 2
n_row_alts       = hl.len(vd.alleles) - 1
n_local_alts     = hl.max(m - 1, 0)
ploidy_known     = hl.is_defined(vd.LGT)
unknown_ploidy   = ~ploidy_known & hl.is_defined(vd.LPL)

subset_la = is_multi_row & (n_local_alts < n_row_alts)
suspect = unknown_ploidy & subset_la

counts = vd.aggregate_entries(hl.struct(
    unknown_multi = hl.agg.count_where(unknown_ploidy & is_multi_row),
    suspect       = hl.agg.count_where(suspect),
))
print(f"Unknown-ploidy entries on multi-ALT rows: {counts.unknown_multi:,}")
print(f"…of which have only a subset of ALTs in LA: {counts.suspect:,}")

# --- peek (recompute derived fields in entries Table 'e') ---
e = vd.filter_entries(suspect).entries()
e = e.select(
    'LA','LGT','LPL','GQ',
    n_row_alts = hl.len(e.alleles) - 1,
    n_local_alts = hl.max(hl.len(hl.or_else(e.LA, hl.empty_array(hl.tint32))) - 1, 0),
    m = hl.len(hl.or_else(e.LA, hl.empty_array(hl.tint32))),
    lenLPL = hl.len(e.LPL),
)
e.show(5)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Unknown-ploidy entries on multi-ALT rows: 870,860
?of which have only a subset of ALTs in LA: 868,212
+---------------+
| locus         |
+---------------+
| locus<GRCh38> |
+---------------+
| chr1:54720    |
| chr1:54720    |
| chr1:66218    |
| chr1:66390    |
| chr1:83829    |
+---------------+

+------------------------------------------------------------------------------+
| alleles                                                                      |
+------------------------------------------------------------------------------+
| array<str>                                                                   |
+------------------------------------------------------------------------------+
| ["CTTTCTTTCTTTCTTTCT","ATTTCTTTCTTTCTTTCT","C","CTCTTTCTTTCT","CTCTTTCTTT... |
| ["CTTTCTTTCTTTCTTTCT","ATTTCTTTCTTTCTTTCT","C","CTCTTTCTTTCT","CTCTTTCTTT... |
| ["AATATATATTATATAATATATATTATATTATATAATATATAATATAAATATAATATAAATT","A","AAT... |
| ["TAATA","AAATA","T","TAAATA","TATA","TATATTATATA

In [12]:
# 2,648 are the cases where, even though ploidy is unknown (LGT is missing) and the row is multi-ALT, the sample’s local allele set LA actually includes all ALTs at that row.
vd = vds.variant_data
la  = hl.or_else(vd.LA, hl.empty_array(hl.tint32))
m   = hl.len(la)
lenLPL = hl.len(vd.LPL)

is_multi = hl.len(vd.alleles) > 2
n_row_alts   = hl.len(vd.alleles) - 1
n_local_alts = hl.max(m - 1, 0)

unknown = ~hl.is_defined(vd.LGT) & hl.is_defined(vd.LPL)

exp_hap = m
exp_dip = (m * (m + 1)) // 2

full_la = unknown & is_multi & (n_local_alts == n_row_alts)

summary = vd.aggregate_entries(hl.struct(
    total = hl.agg.count_where(full_la),
    hap   = hl.agg.count_where(full_la & (lenLPL == exp_hap)),
    dip   = hl.agg.count_where(full_la & (lenLPL == exp_dip)),
    rgq_miss = hl.agg.count_where(full_la & hl.is_missing(vd.RGQ))
))
print(summary)

vd.filter_entries(full_la).entries().select('LA','LGT','LPL','GQ','RGQ').show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Struct(total=2648, hap=10, dip=2638, rgq_miss=0)
+---------------+
| locus         |
+---------------+
| locus<GRCh38> |
+---------------+
| chr1:181328   |
| chr1:631059   |
| chr1:1269203  |
| chr1:2121543  |
| chr1:2121543  |
+---------------+

+------------------------------------------------------------------------------+
| alleles                                                                      |
+------------------------------------------------------------------------------+
| array<str>                                                                   |
+------------------------------------------------------------------------------+
| ["G","A","GCAGGCGCAGAGACACATGCTAGCGCGTCCAGGGGAGGAGGCGTGGCA"]                 |
| ["A","ACCC","ACCCC","ACCCCCC","ACCCCCCCC","ACCCCCCCCC","ACCCCCCCCCC"]        |
| ["G","C","GGGACCCCAAGGCCCCTCAGCCACACCAGAGACTGGGGAGAGGGGTGGTGATCATCAAGCAAG... |
| ["TATCA","T","TA"]                                                           |
| ["TATCA","T","TA"]   

In [13]:
vd = vds.variant_data
la = hl.or_else(vd.LA, hl.empty_array(hl.tint32))
full_la = (~hl.is_defined(vd.LGT)) & hl.is_defined(vd.LPL) & (hl.len(vd.alleles) > 2) & \
          (hl.max(hl.len(la)-1, 0) == (hl.len(vd.alleles)-1))

# LA is unique, in-range, and non-empty?
ok_full_la = vd.aggregate_entries(hl.struct(
    dup = hl.agg.count_where(full_la & (hl.len(hl.set(la)) != hl.len(la))),
    oob = hl.agg.count_where(full_la & hl.any(lambda a: (a < 0) | (a >= hl.len(vd.alleles)), la)),
))
print(ok_full_la)  # should be {'dup': 0, 'oob': 0}


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Struct(dup=0, oob=0)

In [14]:
# minimal, targeted sanitize → split

# 0) Start from your VDS
vd = vds.variant_data

# 1) Define the exact at-risk condition
is_multi_row   = hl.len(vd.alleles) > 2

la             = hl.or_else(vd.LA, hl.empty_array(hl.tint32))
n_local_alts   = hl.max(hl.len(la) - 1, 0)        # local ALTs present in this entry (exclude REF)
n_row_alts     = hl.len(vd.alleles) - 1           # total ALTs at the row (exclude REF)

unknown_ploidy = ~hl.is_defined(vd.LGT)           # LGT missing → ploidy unknown
subset_la      = n_local_alts < n_row_alts        # sample's LA doesn't cover all row ALTs

suspect = is_multi_row & unknown_ploidy & subset_la

# 2) (Optional) quantify how many we’ll touch
n_total_defined = vd.aggregate_entries(hl.agg.count_where(hl.is_defined(vd.LPL)))
n_suspect = vd.aggregate_entries(hl.agg.count_where(hl.is_defined(vd.LPL) & suspect))
print(f"LPL-defined entries: {n_total_defined:,}")
print(f"Sanitizing (unknown ploidy + multi-ALT + subset LA): {n_suspect:,} "
      f"({(n_suspect / n_total_defined * 100 if n_total_defined else 0):.6f}% of LPL-defined)")

# 3) Sanitize ONLY those entries (set LPL to missing so split won’t try to build PL)
vd_clean = vd.annotate_entries(
    LPL = hl.cond(suspect, hl.missing(hl.tarray(hl.tint32)), vd.LPL)
)

# 4) Reassemble a VDS and split safely
vds_clean = hl.vds.VariantDataset(vds.reference_data, vd_clean)

# Drop REF/ALT pairs that would move after min-rep (avoid rare locus-change failures)
vds_bi = hl.vds.split_multi(vds_clean, filter_changed_loci=True)

# 5) Quick sanity checks
vd_bi = vds_bi.variant_data
n_multi_after = vd_bi.aggregate_rows(hl.agg.count_where(hl.len(vd_bi.alleles) > 2))
print(f"Multiallelic rows after split: {n_multi_after:,}")

# (Optional) confirm the sanitized entries are a tiny slice by re-checking the predicate post-split
# (Now rows are biallelic; this is just to show that nothing else blew up.)
n_rows = vd_bi.count_rows()
n_cols = vd_bi.count_cols()
print(f"Biallelic MT shape: rows={n_rows:,}, cols={n_cols:,}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

LPL-defined entries: 17,691,041,658
Sanitizing (unknown ploidy + multi-ALT + subset LA): 868,212 (0.004908% of LPL-defined)

In [15]:
# =============================================================================
# Minimal, targeted sanitize → split (heavily commented)
# -----------------------------------------------------------------------------
# Context / goal
#  - We want to run `hl.vds.split_multi` on a sparse VDS without hitting:
#        "HailException: PL cannot have missing elements"
#  - That error can occur on *some* entries at multi-ALT rows where:
#       (a) the entry's ploidy is unknown (LGT is missing), and
#       (b) the entry's local allele set (LA) is only a *subset* of the row's ALTs.
#    For a split ALT that is *not in* LA, the splitter cannot fill all 3 biallelic
#    PL cells (0/0, 0/1, 1/1) → a missing element → gq_from_pl(PL) fails.
#
# Strategy
#  - Precisely identify those risky entries and set LPL to missing *only* there.
#  - During split, Hail then skips PL construction for those entries and falls back
#    to the existing GQ (`or_else(gq_from_pl(PL), old_GQ)`), so no crash.
#  - This preserves >99.99% of likelihoods and avoids touching safe entries.
#
# Notes
#  - This is a *sparse* workflow. LA and LPL are defined per-entry in the sample's
#    local allele space; the row's global allele list can be much larger.
#  - We keep reference_data unchanged; only variant_data is edited.
# =============================================================================

# --- 0) Start from your VDS ---------------------------------------------------
# Assumes you already have `vds` in scope (a hail.vds.VariantDataset)
vd = vds.variant_data  # the sparse MatrixTable containing variant rows and entries

# --- 1) Define the exact at-risk condition -----------------------------------
# 1a) Multi-ALT row check:
#     A row is "multi-ALT" if it has > 2 alleles total (REF + ≥2 ALT).
#     We only need to guard multi-ALT rows because single-ALT rows map cleanly
#     to biallelic without ambiguity.
is_multi_row = hl.len(vd.alleles) > 2

# 1b) Local allele arithmetic per-entry:
#     LA is the *local* allele set for this entry (REF-inclusive), often a subset
#     of the row's global ALTs. We want counts of local ALTs vs row ALTs (exclude REF).
la           = hl.or_else(vd.LA, hl.empty_array(hl.tint32))     # ensure a real array
n_local_alts = hl.max(hl.len(la) - 1, 0)                        # (#local ALTs; clip at 0)
n_row_alts   = hl.len(vd.alleles) - 1                           # (#row ALTs)

# 1c) Ploidy unknown if LGT is missing:
#     Without LGT, Hail cannot be sure of haploid vs diploid (or more).
#     That uncertainty is what can make the derived biallelic PL incomplete.
unknown_ploidy = ~hl.is_defined(vd.LGT)

# 1d) "Subset LA" means this entry does not cover all row ALTs:
#     If we split to an ALT that isn't in LA, we cannot fill biallelic PL for this entry.
subset_la = n_local_alts < n_row_alts

# 1e) Combine into the "suspect" predicate:
#     Multi-ALT row + unknown ploidy + subset LA
#     → these are the entries that can create a partial PL upon splitting.
suspect = is_multi_row & unknown_ploidy & subset_la

# --- 2) (Optional) quantify impact before editing -----------------------------
#     This is purely informational: how many LPL-defined entries will we touch?
n_total_defined = vd.aggregate_entries(hl.agg.count_where(hl.is_defined(vd.LPL)))
n_suspect = vd.aggregate_entries(hl.agg.count_where(hl.is_defined(vd.LPL) & suspect))
pct = (n_suspect / n_total_defined * 100) if n_total_defined else 0.0
print(f"LPL-defined entries: {n_total_defined:,}")
print(f"Sanitizing (unknown ploidy + multi-ALT + subset LA): {n_suspect:,} ({pct:.6f}% of LPL-defined)")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

LPL-defined entries: 17,691,041,658
Sanitizing (unknown ploidy + multi-ALT + subset LA): 868,212 (0.004908% of LPL-defined)

In [ ]:
# --- 3) Sanitize ONLY those entries ------------------------------------------
#     Key idea: set LPL to missing *only* at suspect entries.
#     Why it works:
#        - When LPL is missing, split_multi does not try to construct a biallelic PL.
#        - It instead keeps the existing GQ (or_else(gq_from_pl(PL), old_GQ)).
#     We guard with `hl.is_defined(vd.LPL)` to avoid rewriting non-existent LPLs.
vd_clean = vd.annotate_entries(
    LPL = hl.cond(
        suspect & hl.is_defined(vd.LPL),
        hl.missing(hl.tarray(hl.tint32)),  # drop likelihoods only where they cannot inform the split
        vd.LPL
    )
)

# (Optional but recommended for large runs) Checkpoint to cut lineage and avoid recomputation:
# vd_clean = vd_clean.checkpoint("s3://YOUR-BUCKET/tmp/vd_clean.mt", overwrite=True)

# Quick audit: ensure no suspect entry still has LPL defined.
remaining = vd_clean.aggregate_entries(hl.agg.count_where(suspect & hl.is_defined(vd_clean.LPL)))
print(f"LPL still defined at suspect entries (should be 0): {remaining:,}")

# --- 4) Reassemble a VDS and split safely ------------------------------------
# Rebuild a VariantDataset with the original reference side and the edited variant side.
vds_clean = hl.vds.VariantDataset(vds.reference_data, vd_clean)

# Perform the actual multi-allelic split on the variant side.
#  - `filter_changed_loci=True` drops REF/ALT pairs that would move after
#    minimal-representation (left/right trim) instead of erroring. Such cases are rare.
vds_bi = hl.vds.split_multi(vds_clean, filter_changed_loci=True)

# --- 5) Quick sanity checks on the result ------------------------------------
vd_bi = vds_bi.variant_data

# There should be *no* multiallelic rows on the variant side after split.
n_multi_after = vd_bi.aggregate_rows(hl.agg.count_where(hl.len(vd_bi.alleles) > 2))
print(f"Multiallelic rows after split: {n_multi_after:,}")

# Basic shape checks (useful to log for downstream steps).
n_rows = vd_bi.count_rows()
n_cols = vd_bi.count_cols()
print(f"Biallelic MT shape: rows={n_rows:,}, cols={n_cols:,}")

# (Optional) Tiny QC: count SNP rows (fast, allele-only)
# snp_rows = vd_bi.aggregate_rows(hl.agg.count_where(hl.is_snp(vd_bi.alleles[0], vd_bi.alleles[1])))
# print(f"SNP rows (biallelic): {snp_rows:,}")

# (Optional) Persist the final VDS
# out_vds = "s3://YOUR-BUCKET/path/my_vds.biallelic.vds"
# vds_bi.write(out_vds, overwrite=True)
# print("Wrote biallelic VDS to:", out_vds)

# =============================================================================
# End of pipeline.
# Summary:
#  - We sanitized only those entries that cannot support a complete biallelic PL
#    upon split (multi-ALT, unknown ploidy, subset LA).
#  - We preserved all other LPLs and left reference_data untouched.
#  - We split with `filter_changed_loci=True` for robust, production-safe behavior.
# =============================================================================

In [12]:
# =============================================================================
# Split a sparse VDS safely by *filtering out* the tiny set of entries that
# cause:  HailException: PL cannot have missing elements
#
# Why this fixes the crash
# ------------------------
# During sparse split, Hail builds a temporary biallelic PL (3 cells) per
# split ALT in order to compute GQ as: GQ = or_else(gq_from_pl(PL), old_GQ).
# If any element of that *temporary* PL is missing, gq_from_pl() throws before
# old_GQ can be used. This happens at multi-ALT rows when LGT (ploidy) is
# missing for an entry — the splitter can’t fully populate 0/0, 0/1, 1/1.
#
# Solution here: filter out exactly those troublesome entries *before* split:
#     risky = (row has ≥2 ALTs) AND (entry LGT is missing)
#
# This avoids constructing the problematic temporary PL entirely.
# We keep >99.99% of entries/calls; impact on downstream metrics is negligible.
#
# Notes
# -----
# - We only touch the variant side (MatrixTable); reference side remains intact.
# - We persist the split VDS before QC to avoid recomputation during counts.
# - Optional extras (checkpointing, per-contig audit) are included as comments.
# =============================================================================


# --- 0) Inputs in scope -------------------------------------------------------
# Assumes you already have:
#   vds: hl.vds.VariantDataset (sparse)
#   vds_prefix / vds_split_multi_uri: paths to write outputs
vd = vds.variant_data

# --- 1) Define the exact, robust guard for “risky-for-split” entries ----------
def risky_expr(mt: hl.MatrixTable) -> hl.expr.BooleanExpression:
    """
    Return a boolean expression on 'mt' that is True exactly for entries that
    are known to cause partial-PL issues during sparse split:

        - multi-ALT row:     len(alleles) > 2
        - unknown ploidy:    LGT is missing at this entry

    This covers both:
        (a) entries whose LA is a subset of row ALTs, and
        (b) the small leftover bucket where LA covers all ALTs but LGT is still NA.
    """
    is_multi_row = hl.len(mt.alleles) > 2   # REF + ≥2 ALTs
    unknown_pl   = ~hl.is_defined(mt.LGT)   # ploidy unknown if LGT is missing
    return is_multi_row & unknown_pl

risky_vd = risky_expr(vd)

# --- 2) Log counts for provenance --------------------------------------------
# Count of defined entries in the sparse matrix:
total_entries = vd.aggregate_entries(hl.agg.count())

# How many entries we will drop (tiny slice, e.g. ~0.005% in your 1KG run)
n_risky = vd.aggregate_entries(hl.agg.count_where(risky_vd))
print(f"Total entries: {total_entries:,}")
print(f"Entries to drop (multi-ALT row & unknown ploidy): {n_risky:,} "
      f"({(n_risky / total_entries * 100 if total_entries else 0):.6f}% of all entries)")

# [Optional] Per-contig audit (cheap, helpful for logs)
# by_contig = (vd
#     .group_rows_by(contig=vd.locus.contig)
#     .aggregate_entries(n_risky=hl.agg.count_where(risky_vd))
#     .rows()
#     .order_by(hl.desc(hl.ref('n_risky'))))  # or hl.desc(by_contig.n_risky) after assignment

# by_contig.show(10)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total entries: 17,691,273,109
Entries to drop (multi-ALT row & unknown ploidy): 870,860 (0.004923% of all entries)

In [13]:
# --- 3) Filter out only the risky entries ------------------------------------
# This is the core fix: remove the entries that would crash the split.
# We do NOT edit arrays (PL/GP/LPL), we just drop the offending entries entirely.
vd_filt = vd.filter_entries(~risky_vd)

# [Optional] Checkpoint to cut lineage on large runs (saves time/memory later).
# vd_filt_uri = vds_prefix + "vd_entries_filtered.mt"
vd_filt = vd_filt.checkpoint(vd_filt_uri, overwrite=True)

# Sanity: recompute the predicate on the *filtered* MT and ensure none remain.
left = vd_filt.aggregate_entries(hl.agg.count_where(risky_expr(vd_filt)))
print(f"Remaining risky entries (expected 0): {left:,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Remaining risky entries (expected 0): 0
2025-08-19 07:39:46.213 Hail: INFO: wrote matrix table with 156228032 rows and 3205 columns in 5767 partitions to s3://precise-scratch/goypav/1KG/VDS/vd_entries_filtered.mt

In [14]:
# --- 4) Reassemble a VDS and split safely ------------------------------------
# Recreate a VariantDataset using:
#   - original reference side (unchanged)
#   - the filtered variant side (our vd_filt)
vds_filt = hl.vds.VariantDataset(vds.reference_data, vd_filt)

# Perform the sparse multi-allelic split.
# `filter_changed_loci=True` drops the extremely rare REF/ALT pairs that would
# move after minimal representation (left/right trimming) instead of erroring.
vds_bi = hl.vds.split_multi(vds_filt, filter_changed_loci=True)

# --- 5) Persist BEFORE QC to avoid recomputation ------------------------------
# vds_split_multi_uri = vds_prefix + "1000genomes_combined.n3205.splitmulti.vds"
vds_bi.write(vds_split_multi_uri, overwrite=True)
print("Wrote biallelic VDS to:", vds_split_multi_uri)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
Error summary: HailException: PL cannot have missing elements.
------------
Hail stack trace:
  File "/mnt/yarn/usercache/livy/appcache/application_1755586145331_0001/container_1755586145331_0001_01_000001/tmp/9246490645552563023", line 744, in <module>
    sys.exit(main())

  File "/mnt/yarn/usercache/livy/appcache/application_1755586145331_0001/container_1755586145331_0001_01_000001/tmp/9246490645552563023", line 716, in main
    response = handler(content)

  File "/mnt/yarn/usercache/livy/appcache/application_1755586145331_0001/container_1755586145331_0001_01_000001/tmp/9246490645552563023", line 326, in execute_request
    result = node.execute()

  File "/mnt/yarn/usercache/livy/appcache/application_1755586145331_0001/container_1755586145331_0001_01_000001/tmp/9246490645552563023", line 232, in execute
    exec(code, global_dict)

  File "<stdin>", line 10, in <module>

  File "/usr/local/lib/python3.9/site-packages/hail/vds/methods.py", line 645, in spl

In [ ]:
# # --- 6) Quick sanity checks on the persisted result ---------------------------
# # Re-read from storage (cheap; does not redo the split).
# vds_bi = hl.vds.read_vds(vds_split_multi_uri)
# vd_bi  = vds_bi.variant_data

# # After a successful split, there should be no multi-allelic rows left.
# n_multi_after = vd_bi.aggregate_rows(hl.agg.count_where(hl.len(vd_bi.alleles) > 2))
# print(f"Multiallelic rows after split: {n_multi_after:,}")

# # Basic shape (useful to log; gives downstream an idea of problem size).
# n_rows = vd_bi.count_rows()
# n_cols = vd_bi.count_cols()
# print(f"Biallelic MT shape: rows={n_rows:,}, cols={n_cols:,}")

# # [Optional] Tiny QC: count biallelic SNP rows (allele-only, fast).
# snps = vd_bi.aggregate_rows(hl.agg.count_where(hl.is_snp(vd_bi.alleles[0], vd_bi.alleles[1])))
# print(f"SNP rows (biallelic): {snps:,}")

# # =============================================================================
# # Summary:
# #  - We removed only entries at multi-ALT rows with missing LGT.
# #  - This prevents split from building partial PLs (which crash gq_from_pl).
# #  - The fraction removed is tiny; reference_data is unchanged.
# #  - We used filter_changed_loci=True for robust, production-safe splitting.
# # =============================================================================

In [15]:
# ensure the splitter never even tries to compute PL for entries whose LA does not cover every ALT at multi-ALT rows—regardless of LGT

# Start from your sparse variant side
vd = vds.variant_data

# --- Identify entries that cannot support a complete biallelic PL -------------
# A row is multi-ALT if it has REF + >=2 ALTs
is_multi_row = hl.len(vd.alleles) > 2

# LA is entry-local allele set (REF-inclusive). Many sparse entries see only a subset.
la           = hl.or_else(vd.LA, hl.empty_array(hl.tint32))
n_local_alts = hl.max(hl.len(la) - 1, 0)      # local ALTs in this entry (exclude REF)
n_row_alts   = hl.len(vd.alleles) - 1         # total ALTs at the row (exclude REF)

# If this entry's LA doesn't include *all* ALTs present at the row, then for at least
# one split ALT, the temporary biallelic PL would have NA cells → gq_from_pl would crash.
subset_la_any = is_multi_row & (n_local_alts < n_row_alts)

# --- Log scope before editing --------------------------------------------------
total_entries = vd.aggregate_entries(hl.agg.count())
n_subset      = vd.aggregate_entries(hl.agg.count_where(subset_la_any))
print(f"Total entries: {total_entries:,}")
print(f"Entries with subset LA on multi-ALT rows: {n_subset:,} "
      f"({(n_subset/total_entries*100 if total_entries else 0):.6f}% of all entries)")
# too many Entries with subset LA on multi-ALT rows: 9,148,398,925 (51.711366% of all entries)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total entries: 17,691,273,109
Entries with subset LA on multi-ALT rows: 9,148,398,925 (51.711366% of all entries)

In [ ]:
# # --- Sanitize exactly those entries: drop only the likelihood arrays -----------
# # Rationale:
# #   - If PL/GP/LPL are missing, the splitter's 'pl' is missing (not partially defined),
# #     so hl.gq_from_pl(pl) -> missing and it cleanly falls back to old_entry.GQ.
# #   - We keep the entry (DP, AD-like fields, GQ, etc.), we only remove likelihoods
# #     where they cannot be faithfully mapped for *every* split ALT.
# annos = {}

# # LPL (sparse non-REF GLs) is common in sparse VDS
# if "LPL" in vd.entry.dtype.fields:
#     annos["LPL"] = hl.cond(subset_la_any & hl.is_defined(vd.LPL),
#                            hl.missing(hl.tarray(hl.tint32)),
#                            vd.LPL)

# # GP (normalized GLs) may exist in some callsets
# if "GP" in vd.entry.dtype.fields:
#     annos["GP"] = hl.cond(subset_la_any & hl.is_defined(vd.GP),
#                           hl.missing(hl.tarray(hl.tfloat64)),
#                           vd.GP)

# # PL (phred GLs) may or may not be present; sanitize if it is
# if "PL" in vd.entry.dtype.fields:
#     annos["PL"] = hl.cond(subset_la_any & hl.is_defined(vd.PL),
#                           hl.missing(hl.tarray(hl.tint32)),
#                           vd.PL)

# vd_sanit = vd.annotate_entries(**annos)

# # Optional: checkpoint to cut lineage on big runs
# # vd_sanit_uri = vds_prefix + "vd_likelihoods_sanitized.mt"
# # vd_sanit = vd_sanit.checkpoint(vd_sanit_uri, overwrite=True)

# # Quick audit: ensure no sanitized entries still carry any likelihood array
# def any_lh_defined(mt):
#     conds = []
#     if "LPL" in mt.entry.dtype.fields: conds.append(hl.is_defined(mt.LPL))
#     if "GP"  in mt.entry.dtype.fields: conds.append(hl.is_defined(mt.GP))
#     if "PL"  in mt.entry.dtype.fields: conds.append(hl.is_defined(mt.PL))
#     return hl.any(lambda x: x, hl.array(conds)) if conds else hl.literal(False)

# leftover = vd_sanit.aggregate_entries(
#     hl.agg.count_where(subset_la_any & any_lh_defined(vd_sanit))
# )
# print(f"Sanitized entries still holding any PL/GP/LPL (should be 0): {leftover:,}")

In [ ]:
# # --- Reassemble VDS and split -------------------------------------------------
# vds_sanit = hl.vds.VariantDataset(vds.reference_data, vd_sanit)

# # Note: filter_changed_loci=True discards the vanishingly small set of REF/ALT pairs
# # that would move under minimal representation (safer than erroring).
# vds_bi = hl.vds.split_multi(vds_sanit, filter_changed_loci=True)

# # Persist BEFORE QC to avoid recomputation during counts
# # vds_split_multi_uri = vds_prefix + "1000genomes_combined.n3205.splitmulti.vds"
# vds_bi.write(vds_split_multi_uri, overwrite=True)
# print("Wrote biallelic VDS to:", vds_split_multi_uri)

In [ ]:
# # --- Quick sanity checks on the persisted result ------------------------------
# vds_bi = hl.vds.read_vds(vds_split_multi_uri)
# vd_bi  = vds_bi.variant_data

# n_multi_after = vd_bi.aggregate_rows(hl.agg.count_where(hl.len(vd_bi.alleles) > 2))
# print(f"Multiallelic rows after split: {n_multi_after:,}")

# n_rows = vd_bi.count_rows()
# n_cols = vd_bi.count_cols()
# print(f"Biallelic MT shape: rows={n_rows:,}, cols={n_cols:,}")

# Narrow down which rows are failing

In [10]:
# Cell 1 — Helpers and derived features (all on one MT)

vd  = vds.variant_data
vdr = vds.reference_data

def with_entry_features(mt: hl.MatrixTable) -> hl.MatrixTable:
    """
    Add derived entry/row features we’ll need for profiling.
    - multi-ALT row
    - local/global ALT counts (subset-LA)
    - LGT presence, ploidy, hom-ref flags
    - RGQ/GQ presence
    - which likelihood arrays exist (LPL/PL/GP) and their lengths
    """
    la = hl.or_else(mt.LA, hl.empty_array(hl.tint32))
    m_local = hl.max(hl.len(la) - 1, 0)          # local ALT count (exclude REF)
    m_row   = hl.len(mt.alleles) - 1             # row ALT count   (exclude REF)

    have_LGT   = hl.is_defined(mt.LGT)
    ploidy     = hl.or_missing(have_LGT, mt.LGT.ploidy)
    is_hap     = have_LGT & (ploidy == 1)
    is_dip     = have_LGT & (ploidy == 2)
    # “hom-ref” for haploid/diploid:
    # haploid: 0 ; diploid: 0/0
    is_homref  = have_LGT & (
        (is_hap & (mt.LGT.one_hot_alleles().get(0, 0) == 1)) |
        (is_dip & mt.LGT.is_hom_ref())
    )

    feat = dict(
        is_multi_row = hl.len(mt.alleles) > 2,
        m_local_alts = m_local,
        m_row_alts   = m_row,
        subset_LA    = (m_local < m_row),
        have_LGT     = have_LGT,
        ploidy       = ploidy,
        is_hap       = is_hap,
        is_dip       = is_dip,
        is_homref    = is_homref,
        have_RGQ     = hl.is_defined(mt.RGQ) if "RGQ" in mt.entry.dtype.fields else hl.literal(False),
        have_GQ      = hl.is_defined(mt.GQ)  if "GQ"  in mt.entry.dtype.fields else hl.literal(False),

        have_LPL     = hl.is_defined(mt.LPL) if "LPL" in mt.entry.dtype.fields else hl.literal(False),
        have_PL      = hl.is_defined(mt.PL)  if "PL"  in mt.entry.dtype.fields else hl.literal(False),
        have_GP      = hl.is_defined(mt.GP)  if "GP"  in mt.entry.dtype.fields else hl.literal(False),

        len_LPL      = hl.or_missing(hl.is_defined(mt.LPL), hl.len(mt.LPL)) if "LPL" in mt.entry.dtype.fields else hl.missing(hl.tint32),
        len_PL       = hl.or_missing(hl.is_defined(mt.PL),  hl.len(mt.PL))  if "PL"  in mt.entry.dtype.fields else hl.missing(hl.tint32),
        len_GP       = hl.or_missing(hl.is_defined(mt.GP),  hl.len(mt.GP))  if "GP"  in mt.entry.dtype.fields else hl.missing(hl.tint32),
    )
    return mt.annotate_entries(**feat)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
# Cell 2 — A “smoke split” function that tests a slice and returns rows that fail

def try_split_and_flag_rows(vdr, vd_slice: hl.MatrixTable) -> hl.Table:
    """
    Attempt sparse split on a MatrixTable slice and FORCE entry evaluation.
    If it fails, mark ALL rows in that slice as 'suspected_failing'.
    If it succeeds, return an empty table.
    """
    try:
        vtmp   = hl.vds.VariantDataset(vdr, vd_slice)
        vsplit = hl.vds.split_multi(vtmp, filter_changed_loci=True).variant_data

        # FORCE the per-entry transform to run by referencing an entry field.
        # GQ is ideal because the splitter computes it via gq_from_pl(PL).
        if "GQ" in vsplit.entry.dtype.fields:
            _ = vsplit.aggregate_entries(hl.agg.count_where(hl.is_defined(vsplit.GQ)))
        else:
            # Fallback: any entry expression will force the map; use PL/LPL/GP if present.
            choices = [f for f in ("PL", "LPL", "GP") if f in vsplit.entry.dtype.fields]
            if choices:
                f = choices[0]
                _ = vsplit.aggregate_entries(hl.agg.count_where(hl.is_defined(vsplit[f])))
            else:
                # Last resort: count entries still forces entry iteration on vsplit
                _ = vsplit.aggregate_entries(hl.agg.count())

        # success -> no failing rows in this slice
        return vd_slice.rows().annotate(suspected_failing=False).filter(lambda r: False)

    except Exception:
        # failure -> all rows in this slice are suspicious
        return vd_slice.rows().annotate(suspected_failing=True)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
# Cell 3 — Narrow to small failing windows (re-usable from earlier, but returns row masks)
def contig_limits(vd: hl.MatrixTable, contig: str):
    mts = vd.filter_rows(vd.locus.contig == contig)
    lim = mts.aggregate_rows(hl.struct(
        minp = hl.agg.min(mts.locus.position),
        maxp = hl.agg.max(mts.locus.position),
    ))
    return lim.minp, lim.maxp

def slice_vd(vd: hl.MatrixTable, contig: str, start: int, end: int):
    return vd.filter_rows((vd.locus.contig == contig) &
                          (vd.locus.position >= start) &
                          (vd.locus.position <= end))

def bisect_fail_windows(vdr, vd, contig, start, end, min_window=200_000, acc=None):
    if acc is None:
        acc = []
    vd_win = slice_vd(vd, contig, start, end)
    try:
        # IMPORTANT: this calls the entry-forcing version above
        _ = try_split_and_flag_rows(vdr, vd_win)
        # if no exception, the slice is clean
        return acc
    except Exception:
        # should not reach here because we catch inside try_split_and_flag_rows,
        # but keep the defensive split in case of non-standard failures
        pass

    # Explicitly use the return value to decide whether to bisect
    flagged = try_split_and_flag_rows(vdr, vd_win)
    # If we flagged it as failing, bisect or record window
    if end - start <= min_window:
        acc.append((contig, start, end))
        return acc
    mid = (start + end) // 2
    bisect_fail_windows(vdr, vd, contig, start,  mid, min_window, acc)
    bisect_fail_windows(vdr, vd, contig, mid+1, end, min_window, acc)
    return acc

# 1) find failing contigs (now forcing entry eval per contig)
contigs = sorted(vd.aggregate_rows(hl.agg.collect_as_set(vd.locus.contig)))
failing_contigs = []
for c in contigs:
    vd_c = vd.filter_rows(vd.locus.contig == c)
    flagged = try_split_and_flag_rows(vds.reference_data, vd_c)
    # If the helper returned a non-empty table, this contig failed
    if flagged.count() > 0:
        failing_contigs.append(c)
print("Failing contigs:", failing_contigs)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Failing contigs: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chrM', 'chrX', 'chrY']

In [13]:
# --- Drive the bisection on failing contigs -----------------------------------

# If you don't already have 'failing_contigs' from the entry-evaluating check, recompute:
# failing_contigs = []
# for c in sorted(vd.aggregate_rows(hl.agg.collect_as_set(vd.locus.contig))):
#     if not split_slice_ok(vds.reference_data, vd.filter_rows(vd.locus.contig == c)):
#         failing_contigs.append(c)

print("Failing contigs:", failing_contigs)

def merge_windows(wins):
    """Merge overlapping/contiguous (contig, start, end) tuples."""
    by_c = {}
    for c,s,e in wins:
        by_c.setdefault(c, []).append((s,e))
    merged = []
    for c, segs in by_c.items():
        segs.sort()
        cur_s, cur_e = segs[0]
        for s,e in segs[1:]:
            if s <= cur_e + 1:          # overlap or touch
                cur_e = max(cur_e, e)
            else:
                merged.append((c, cur_s, cur_e))
                cur_s, cur_e = s, e
        merged.append((c, cur_s, cur_e))
    # sort by contig then start
    return sorted(merged, key=lambda x: (x[0], x[1]))

fail_windows = []
for c in failing_contigs:
    # contig may be totally filtered in upstream steps; skip if no bounds
    lo, hi = contig_limits(vd, c)
    if (lo is None) or (hi is None) or (hi < lo):
        continue
    # bisect down to ~200kb windows (tune as needed)
    fail_windows.extend(
        bisect_fail_windows(vds.reference_data, vd, c, int(lo), int(hi), min_window=200_000)
    )

# Coalesce overlapping windows; keeps output readable
fail_windows = merge_windows(fail_windows)

print(f"Identified {len(fail_windows)} failing windows")
for w in fail_windows[:15]:
    print(w)

# Optional: profile the first few windows to see common traits
def window_profile(vd, contig, start, end):
    mts = slice_vd(vd, contig, start, end)
    la  = hl.or_else(mts.LA, hl.empty_array(hl.tint32))
    n_local = hl.max(hl.len(la) - 1, 0)
    n_row   = hl.len(mts.alleles) - 1
    is_multi = hl.len(mts.alleles) > 2
    subset   = is_multi & (n_local < n_row)
    return mts.aggregate_entries(hl.struct(
        entries=hl.agg.count(),
        lgt_missing=hl.agg.count_where(hl.is_missing(mts.LGT)),
        multi=hl.agg.count_where(is_multi),
        subsetLA=hl.agg.count_where(subset),
        LPL_def=hl.agg.count_where(hl.is_defined(mts.LPL)),
        GP_def=hl.agg.count_where(hl.is_defined(mts.GP)) if "GP" in mts.entry.dtype.fields else hl.int64(0)
    ))

for c,s,e in fail_windows[:5]:
    stats = window_profile(vd, c, s, e)
    print(f"{c}:{s}-{e}  ->  {stats}")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkStatementCancellationFailedException: Interrupted by user but Livy failed to cancel the Spark statement. The Livy session might have become unusable.

In [ ]:
# 2) for each failing contig, bisect into failing windows (entry-evaluating)
fail_windows = []
for c in failing_contigs:
    lo, hi = contig_limits(vd, c)
    fail_windows.extend(bisect_fail_windows(vds.reference_data, vd, c, lo, hi, min_window=200_000))
print("Failing windows:", fail_windows)

In [ ]:
# Cell 4 — Extract entries only inside failing windows and attach our features
# Build a single boolean row-mask for all failing windows
row_in_fail = hl.literal(False)
for (c, s, e) in fail_windows:
    row_in_fail = row_in_fail | ((vd.locus.contig == c) &
                                 (vd.locus.position >= s) &
                                 (vd.locus.position <= e))

vd_fail = vd.filter_rows(row_in_fail)
vd_fail = with_entry_features(vd_fail)

# Quick sizes
n_rows_fail = vd_fail.count_rows()
n_ent_fail  = vd_fail.aggregate_entries(hl.agg.count())
print(f"Rows in failing windows: {n_rows_fail:,}, entries there: {n_ent_fail:,}")

In [ ]:
# Cell 5 — Profile traits of entries in failing windows
# Common hypotheses we’ll quantify:
# H1: multi-ALT row AND LGT missing AND subset-LA (classic)
h1 = vd_fail.is_multi_row & ~vd_fail.have_LGT & vd_fail.subset_LA

# H2: multi-ALT row AND LGT present-homref AND subset-LA AND RGQ missing
#     (ref-only entry at a multi-ALT row, no RGQ to fabricate biallelic PL)
h2 = vd_fail.is_multi_row & vd_fail.have_LGT & vd_fail.is_homref & vd_fail.subset_LA & ~vd_fail.have_RGQ

# H3: multi-ALT row AND LGT present-homref AND subset-LA AND *no* LPL/PL/GP either
#     (nothing to build PL from; RGQ may or may not exist)
h3 = (vd_fail.is_multi_row & vd_fail.have_LGT & vd_fail.is_homref & vd_fail.subset_LA &
      ~vd_fail.have_LPL & ~vd_fail.have_PL & ~vd_fail.have_GP)

# H4: multi-ALT row AND LGT present (any) AND ANY of the likelihood arrays have
#     internal missing elements (paranoia check; you earlier saw 0, but we recheck here)
h4 = (vd_fail.is_multi_row & vd_fail.have_LGT &
      (
          (vd_fail.have_LPL & hl.any(lambda x: hl.is_missing(x), vd_fail.LPL)) |
          (vd_fail.have_PL  & hl.any(lambda x: hl.is_missing(x), vd_fail.PL))  |
          (vd_fail.have_GP  & hl.any(lambda x: hl.is_missing(x), vd_fail.GP))
      ))

counts = vd_fail.aggregate_entries(hl.struct(
    total       = hl.agg.count(),
    H1          = hl.agg.count_where(h1),
    H2          = hl.agg.count_where(h2),
    H3          = hl.agg.count_where(h3),
    H4          = hl.agg.count_where(h4),
))
print(counts)


In [ ]:
# Cell 6 — Show concrete examples for each hypothesis
def show_examples(cond, title, n=10):
    print(f"\n=== {title} (n={vd_fail.aggregate_entries(hl.agg.count_where(cond)):,}) ===")
    cols = dict(
        locus = vd_fail.locus,
        alleles = vd_fail.alleles,
        s  = vd_fail.s,
        LA = vd_fail.LA,
        LGT = vd_fail.LGT,
        GQ = vd_fail.GQ if "GQ" in vd_fail.entry.dtype.fields else hl.missing(hl.tint32),
        RGQ = vd_fail.RGQ if "RGQ" in vd_fail.entry.dtype.fields else hl.missing(hl.tint32),
        is_multi_row = vd_fail.is_multi_row,
        subset_LA = vd_fail.subset_LA,
        have_LGT = vd_fail.have_LGT,
        is_homref = vd_fail.is_homref,
        have_RGQ = vd_fail.have_RGQ,
        have_LPL = vd_fail.have_LPL,
        have_PL  = vd_fail.have_PL,
        have_GP  = vd_fail.have_GP,
        m_local_alts = vd_fail.m_local_alts,
        m_row_alts   = vd_fail.m_row_alts,
    )
    if "LPL" in vd_fail.entry.dtype.fields:
        cols["LPL"] = vd_fail.LPL
    if "PL" in vd_fail.entry.dtype.fields:
        cols["PL"] = vd_fail.PL
    if "GP" in vd_fail.entry.dtype.fields:
        cols["GP"] = vd_fail.GP

    vd_fail.filter_entries(cond).entries().select(**cols).show(n)

show_examples(h1, "H1: multi-ALT & LGT missing & subset-LA", n=10)
show_examples(h2, "H2: multi-ALT & hom-ref & subset-LA & RGQ missing", n=10)
show_examples(h3, "H3: multi-ALT & hom-ref & subset-LA & no LPL/PL/GP", n=10)
show_examples(h4, "H4: multi-ALT & LGT present & internal NA inside likelihood arrays", n=10)


In [ ]:
# Cell 7 — Build a conditional predicate from what we learned
# Start with the classic cause:
bad = h1

# Promote additional tiny buckets IF they show up in failing windows:
bad = bad | h2   # usually small but important
# bad = bad | h3 # uncomment only if needed based on your counts/examples
# bad = bad | h4 # usually zero, but include if you actually observe >0

n_bad = vd_fail.aggregate_entries(hl.agg.count_where(bad))
print(f"Combined 'bad' predicate in failing windows: {n_bad:,} entries")


In [ ]:
# Cell 8 — Validate the predicate by filtering only inside the windows and smoke-splitting
# Apply the filter ONLY within failing windows, leave the rest of the genome untouched.
mask_fail_rows = row_in_fail
vd_test = vd.annotate_entries(_drop = hl.cond(mask_fail_rows & bad, True, False))
vd_test = vd_test.filter_entries(~vd_test._drop).drop(vd_test._drop)

# Quick smoke split over JUST the failing windows
vd_test_fail = vd_test.filter_rows(row_in_fail)
ok = True
try:
    vtmp = hl.vds.VariantDataset(vdr, vd_test_fail)
    _ = hl.vds.split_multi(vtmp, filter_changed_loci=True).variant_data.count_rows()
    print("[OK] Smoke-split succeeds after conditional drop inside failing windows.")
except Exception as e:
    ok = False
    print("[FAIL] Smoke-split still failing:", e)

# If ok, we likely have the right predicate. Next step: apply to full dataset.


In [ ]:
# Cell 9 — Apply globally (only entries matching ‘bad’, genome-wide) and split
if 'ok' in locals() and ok:
    # Build the same 'bad' predicate on the *full* variant MT
    vd_full = with_entry_features(vd)

    # Recreate H1/H2/... expressions on vd_full
    H1_full = vd_full.is_multi_row & ~vd_full.have_LGT & vd_full.subset_LA
    H2_full = vd_full.is_multi_row & vd_full.have_LGT & vd_full.is_homref & vd_full.subset_LA & ~vd_full.have_RGQ
    # H3_full = ...
    # H4_full = ...

    bad_full = H1_full | H2_full   # add others only if needed based on your profiling

    n_bad_full = vd_full.aggregate_entries(hl.agg.count_where(bad_full))
    n_tot_full = vd_full.aggregate_entries(hl.agg.count())
    print(f"Will drop globally: {n_bad_full:,} entries "
          f"({n_bad_full / n_tot_full * 100:.6f}% of all entries)")

    # Filter & split
    vd_fixed = vd_full.filter_entries(~bad_full)
    vds_fixed = hl.vds.VariantDataset(vdr, vd_fixed.select_entries(*[f for f in vd.entry if f in vd_fixed.entry]))
    vds_bi   = hl.vds.split_multi(vds_fixed, filter_changed_loci=True)

    # Persist BEFORE QC so counts don't recompute the split
    vds_bi.write(vds_split_multi_uri, overwrite=True)
    print("Wrote biallelic VDS to:", vds_split_multi_uri)


In [ ]:
# Cell 10 — Sanity checks
vds_bi = hl.vds.read_vds(vds_split_multi_uri)
vd_bi  = vds_bi.variant_data

n_multi_after = vd_bi.aggregate_rows(hl.agg.count_where(hl.len(vd_bi.alleles) > 2))
print(f"Multiallelic rows after split: {n_multi_after:,}")

n_rows = vd_bi.count_rows()
n_cols = vd_bi.count_cols()
print(f"Biallelic MT shape: rows={n_rows:,}, cols={n_cols:,}")

snps = vd_bi.aggregate_rows(hl.agg.count_where(hl.is_snp(vd_bi.alleles[0], vd_bi.alleles[1])))
print(f"SNP rows (biallelic): {snps:,}")


In [10]:
# Cell 1 — helpers (unchanged)
# import hail as hl
import time

def timeit(label, fn, *args, **kwargs):
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    dt = time.perf_counter() - t0
    print(f"[{label}] {dt:.1f}s")
    return out

def split_slice_ok(vdr, mts: hl.MatrixTable):
    # Force entry evaluation so we actually hit the gq_from_pl path
    try:
        vtmp   = hl.vds.VariantDataset(vdr, mts)
        vsplit = hl.vds.split_multi(vtmp, filter_changed_loci=True)
        _ = vsplit.variant_data.count_rows()
        return True, "ok"
    except Exception as e:
        return False, f"{type(e).__name__}: {str(e).splitlines()[0]}"

def slice_quick_stats(mts: hl.MatrixTable):
    la          = hl.or_else(mts.LA, hl.empty_array(hl.tint32))
    is_multi    = hl.len(mts.alleles) > 2
    missing_lgt = hl.is_missing(mts.LGT)
    n_local     = hl.max(hl.len(la) - 1, 0)
    n_row       = hl.len(mts.alleles) - 1
    subset_la   = is_multi & (n_local < n_row)

    return mts.aggregate_entries(hl.struct(
        entries      = hl.agg.count(),
        multi_rows   = hl.agg.count_where(is_multi),
        lgt_missing  = hl.agg.count_where(missing_lgt),
        subset_LA    = hl.agg.count_where(subset_la),
        lpl_defined  = hl.agg.count_where(hl.is_defined(mts.LPL)),
        gp_defined   = hl.agg.count_where(hl.is_defined(mts.GP)) if "GP" in mts.entry.dtype.fields else hl.int64(0),
    ))


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
# Cell 2 — list contigs
vd = vds.variant_data
contigs = sorted(vd.aggregate_rows(hl.agg.collect_as_set(vd.locus.contig)))
print("Contigs detected:", contigs)
CONTIGS_TO_RUN = contigs  # or a subset like ['chr1','chrX']

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Contigs detected: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chrM', 'chrX', 'chrY']

In [12]:
# Cell 3 — per-contig driver (no coalesce; use persist)
for i, c in enumerate(CONTIGS_TO_RUN, 1):
    print(f"\n=== [{i}/{len(CONTIGS_TO_RUN)}] Contig {c} ===")

    # Build the slice and persist once; avoids recomputing the same slice
    mts = vd.filter_rows(vd.locus.contig == c).persist()  # <-- no coalesce

    stats = timeit(f"{c} stats", slice_quick_stats, mts)
    print(f" entries={stats.entries:,} | multi-ALT rows={stats.multi_rows:,} | "
          f"LGT missing={stats.lgt_missing:,} | subset-LA entries={stats.subset_LA:,} | "
          f"LPL_defined={stats.lpl_defined:,}"
          + (f" | GP_defined={stats.gp_defined:,}" if 'gp_defined' in stats else ""))

    ok, why = timeit(f"{c} split test", split_slice_ok, vds.reference_data, mts)
    print(" split:", "OK" if ok else "FAIL", "|", why)

    # Let Spark evict when needed; no explicit unpersist required


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


=== [1/25] Contig chr1 ===
[chr1 stats] 17.9s
 entries=1,398,365,778 | multi-ALT rows=760,218,764 | LGT missing=70,992 | subset-LA entries=758,030,709 | LPL_defined=1,398,365,778 | GP_defined=1,398,365,778
[chr1 split test] 18.5s
 split: OK | ok

=== [2/25] Contig chr10 ===
[chr10 stats] 16.2s
 entries=871,265,814 | multi-ALT rows=454,425,097 | LGT missing=45,654 | subset-LA entries=453,062,297 | LPL_defined=871,265,814 | GP_defined=871,265,814
[chr10 split test] 15.8s
 split: OK | ok

=== [3/25] Contig chr11 ===
[chr11 stats] 15.8s
 entries=858,594,171 | multi-ALT rows=447,375,252 | LGT missing=31,787 | subset-LA entries=446,051,438 | LPL_defined=858,594,171 | GP_defined=858,594,171
[chr11 split test] 15.0s
 split: OK | ok

=== [4/25] Contig chr12 ===
[chr12 stats] 14.0s
 entries=809,751,667 | multi-ALT rows=425,835,139 | LGT missing=38,329 | subset-LA entries=424,663,196 | LPL_defined=809,751,667 | GP_defined=809,751,667
[chr12 split test] 13.5s
 split: OK | ok

=== [5/25] Contig ch

In [13]:
# Cell 4 — (optional) bisection without coalesce
def contig_limits(vd: hl.MatrixTable, contig: str):
    mts = vd.filter_rows(vd.locus.contig == contig)
    lim = mts.aggregate_rows(hl.struct(
        minp = hl.agg.min(mts.locus.position),
        maxp = hl.agg.max(mts.locus.position),
    ))
    return lim.minp, lim.maxp

def slice_vd(vd: hl.MatrixTable, contig: str, start: int, end: int):
    return vd.filter_rows((vd.locus.contig == contig) &
                          (vd.locus.position >= start) &
                          (vd.locus.position <= end))

def bisect_fail_windows(vdr, vd, contig, start, end, min_window=200_000, acc=None):
    if acc is None:
        acc = []
    mts = slice_vd(vd, contig, start, end)  # <-- no coalesce
    ok, _ = split_slice_ok(vdr, mts)
    if ok:
        return acc
    if end - start <= min_window:
        acc.append((contig, start, end))
        return acc
    mid = (start + end) // 2
    bisect_fail_windows(vdr, vd, contig, start,  mid, min_window, acc)
    bisect_fail_windows(vdr, vd, contig, mid+1, end, min_window, acc)
    return acc

def merge_windows(wins):
    by_c = {}
    for c,s,e in wins:
        by_c.setdefault(c, []).append((s,e))
    merged = []
    for c, segs in by_c.items():
        segs.sort()
        cs, ce = segs[0]
        for s,e in segs[1:]:
            if s <= ce + 1:
                ce = max(ce, e)
            else:
                merged.append((c, cs, ce))
                cs, ce = s, e
        merged.append((c, cs, ce))
    return sorted(merged, key=lambda x: (x[0], x[1]))

# example: bisect one failing contig you saw above
CONTIG_TO_BISECT = "chr1"
lo, hi = contig_limits(vd, CONTIG_TO_BISECT)
print(f"{CONTIG_TO_BISECT} bounds: {lo}-{hi}")

fw = timeit(f"bisect {CONTIG_TO_BISECT}", bisect_fail_windows,
            vds.reference_data, vd, CONTIG_TO_BISECT, int(lo), int(hi), min_window=200_000)
fw = merge_windows(fw)
print(f"Failing windows on {CONTIG_TO_BISECT}: {len(fw)}")
for w in fw[:15]:
    print(" ", w)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

chr1 bounds: 10001-248946420
[bisect chr1] 26.2s
Failing windows on chr1: 0

# Test sparse mt

In [10]:
# ---- settings you may tweak ----
# Small windows on small chromosomes for speed
WINDOW_SIZE = 2_000_000   # 2 Mb
WINDOW_STARTS = [1, 10_000_000]   # two windows per contig
CONTIG_CANDIDATES = ["22", "21", "20"]  # try small chroms first
coalesce_parts = 128  # keep sample write small/fast
hl.set_global_seed(1)

# Output paths
# sparse_sample_allrows_mt_uri = "1000genomes_combined.n3205.sparse.sample_allrows.mt"
# sparse_sample_variants_only_mt_uri = "1000genomes_combined.n3205.sparse.sample_variants_only.mt"

# ---------- 1) Detect contig naming & build intervals ----------
rg_name = vds.variant_data.locus.dtype.reference_genome.name
rg = hl.get_reference(rg_name)

# Peek first row to decide if contigs are 'chr22' or '22'
first_row = vds.variant_data.rows().take(1)[0]
has_chr_prefix = first_row.locus.contig.startswith("chr")

def name_with_prefix(x):
    return f"chr{x}" if has_chr_prefix else x

# Keep only contigs that exist on this reference
contigs_present = set(rg.contigs)
picked_contigs = [c for c in (name_with_prefix(c) for c in CONTIG_CANDIDATES) if c in contigs_present]
if not picked_contigs:
    # fallback: just use the first contig we saw
    picked_contigs = [first_row.locus.contig]

# Build small intervals per contig (cap at contig length if available)
lengths = getattr(rg, "lengths", None)  # dict[str,int] when known
intervals = []
for c in picked_contigs:
    for s in WINDOW_STARTS:
        if lengths and c in lengths:
            e = min(s + WINDOW_SIZE - 1, lengths[c])
            if s > e: 
                continue
        else:
            # if lengths not available, just create an interval; Hail will clamp internally
            e = s + WINDOW_SIZE - 1
        intervals.append(hl.parse_locus_interval(f"{c}:{s}-{e}", rg_name))

print("Using intervals:", [f"{i.start.contig}:{i.start.position}-{i.end.position}" for i in intervals])

# ---------- 2) Slice the VDS to those intervals ----------
vds_slice = hl.vds.filter_intervals(vds, intervals, keep=True)

# ---------- 3) Convert ONLY the slice to merged sparse MT ----------
# Supply a ref_allele_function so we don't need a FASTA for pure-ref rows
ref_fun = lambda locus: hl.missing(hl.tstr)
mt_all = hl.vds.to_merged_sparse_mt(vds_slice, ref_allele_function=ref_fun)

# Variants-only view
mt_var = mt_all.filter_rows(hl.len(mt_all.alleles) > 1)

# Optional: coalesce the *samples* for faster writes
mt_all_sm = mt_all.naive_coalesce(coalesce_parts)
mt_var_sm = mt_var.naive_coalesce(coalesce_parts)

# ---------- 4) Write BOTH smoke tests ----------
mt_all_sm.write(sparse_sample_allrows_mt_uri, overwrite=True)
print("Wrote merged sparse sample (ALL rows) to:", sparse_sample_allrows_mt_uri)

mt_var_sm.write(sparse_sample_variants_only_mt_uri, overwrite=True)
print("Wrote merged sparse sample (VARIANTS only) to:", sparse_sample_variants_only_mt_uri)

# ---------- 5) Read back & quick stats ----------
mt_all_back = hl.read_matrix_table(sparse_sample_allrows_mt_uri)
n_rows_all = mt_all_back.count_rows()
n_cols_all = mt_all_back.count_cols()
n_var_all  = mt_all_back.aggregate_rows(hl.agg.count_where(hl.len(mt_all_back.alleles) > 1))
print(f"[ALL rows] rows: {n_rows_all:,} | cols: {n_cols_all:,} | variants: {n_var_all:,} | pure-ref: {n_rows_all - n_var_all:,}")

mt_var_back = hl.read_matrix_table(sparse_sample_variants_only_mt_uri)
n_rows_var = mt_var_back.count_rows()
n_cols_var = mt_var_back.count_cols()
print(f"[VARIANTS only] rows: {n_rows_var:,} | cols: {n_cols_var:,}")

# Tiny Ti/Tv on the variant-only sample
ref = mt_var_back.alleles[0]; alt = mt_var_back.alleles[1]
is_snp = hl.is_snp(ref, alt)
ti = mt_var_back.aggregate_rows(hl.agg.count_where(is_snp & hl.is_transition(ref, alt)))
tv = mt_var_back.aggregate_rows(hl.agg.count_where(is_snp & ~hl.is_transition(ref, alt)))
if tv > 0:
    print(f"[VARIANTS only] Ti/Tv: {ti/tv:.3f}  (Ti {ti:,}, Tv {tv:,})")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Using intervals: ['<StringExpression of type str>:<Int32Expression of type int32>-<Int32Expression of type int32>', '<StringExpression of type str>:<Int32Expression of type int32>-<Int32Expression of type int32>', '<StringExpression of type str>:<Int32Expression of type int32>-<Int32Expression of type int32>', '<StringExpression of type str>:<Int32Expression of type int32>-<Int32Expression of type int32>', '<StringExpression of type str>:<Int32Expression of type int32>-<Int32Expression of type int32>', '<StringExpression of type str>:<Int32Expression of type int32>-<Int32Expression of type int32>']
Wrote merged sparse sample (ALL rows) to: 1000genomes_combined.n3205.sparse.sample_allrows.mt
Wrote merged sparse sample (VARIANTS only) to: 1000genomes_combined.n3205.sparse.sample_variants_only.mt
[ALL rows] rows: 5,636,342 | cols: 3,205 | variants: 446,222 | pure-ref: 5,190,120
[VARIANTS only] rows: 446,222 | cols: 3,205
[VARIANTS only] Ti/Tv: 1.283  (Ti 225,832, Tv 176,086)
2025-08-14 06

# Test dense mt